In [7]:
import os
import time
import threading
import speech_recognition as sr
from gtts import gTTS
from pydub import AudioSegment
from pydub.playback import play
import tempfile
from langdetect import detect
import googletrans
import weaviate
from weaviate.auth import AuthApiKey
from openai import AzureOpenAI
from langchain.memory import ConversationBufferMemory
from langchain_core.prompts import PromptTemplate
import uuid

In [ ]:
# Configuration
WEAVIATE_CONFIG = {
    "http_host": "localhost",
    "http_port": 8080,
    "grpc_host": "localhost",
    "grpc_port": 50051,
    "secure": False
}


AZURE_CONFIG = {
    "api_key":  "2odd5gymXIKl0N6xUvPEx79xZbVBjJm1SPy6AWLZtNYMw25QoLDHJQQJ99BEACYeBjFXJ3w3AAABACOGQfkw",
    "endpoint": "https://weaviaterag.openai.azure.com/",
    "api_version": "2024-12-01-preview",
    "embedding_deployment": "text-embedding-ada-002",
    "gpt_deployment": "weaviaterag"
}

# Language codes for supported languages
LANGUAGES = {
    "english": "en",
    "malay": "ms",
    "thai": "th",
    "chinese": "zh-CN",
    "tamil": "ta",
    "telugu": "te",
    "singlish": "en" # Default to English for Singlish
}

In [9]:
class WeaviateManager:
    def __init__(self, http_host, http_port, grpc_host, grpc_port, secure=False, azure_config=None):
        self.client = weaviate.connect_to_custom(
            http_host=http_host,
            http_port=http_port,
            grpc_host=grpc_host,
            grpc_port=grpc_port,
            http_secure=secure,
            grpc_secure=secure
        )
        self.azure_client = None
        self.collection_names = []
        self.memory = ConversationBufferMemory(return_messages=True, memory_key="chat_history")
        
        if azure_config:
            self.azure_client = AzureOpenAI(
                azure_endpoint=azure_config["endpoint"],
                api_key=azure_config["api_key"],
                api_version=azure_config["api_version"]
            )
            self.embedding_model = azure_config["embedding_deployment"]
            self.gpt_model = azure_config["gpt_deployment"]
    
    def get_collection_names(self):
        """Get all collection names from Weaviate and store them in self.collection_names"""
        collections = self.client.collections.list_all()
        self.collection_names = collections 

    def search_query_in_collections(self, query_text, top_k=5):
        """Search for query in all collections"""
        query_embedding = self.get_text_embedding(query_text) if self.azure_client else None
        results = []

        if not self.collection_names:
            self.get_collection_names()

        for collection_name in self.collection_names:
            if self.client.collections.exists(collection_name):
                collection = self.client.collections.get(collection_name)
                result = collection.query.near_vector(
                    near_vector=query_embedding,
                    limit=top_k,
                    distance=0.7,
                    return_properties=["title", "content"]
                )
                results.extend(result.objects)
            else:
                print(f"Collection '{collection_name}' does not exist.")

        return self.format_results_with_llm(results, query_text)

    def format_results_with_llm(self, details, user_input):
        """Format results using LLM"""
        chat_history = self.memory.load_memory_variables({})["chat_history"]
        chat_history_str = "\n".join([f"{msg.type}: {msg.content}" for msg in chat_history])
        details_str = "\n\n".join([f"Document {i+1}:\nTitle: {obj.properties['title']}\nContent: {obj.properties['content']}" for i, obj in enumerate(details)])

        prompt_template = PromptTemplate(
            input_variables=["chat_history", "details", "user_input"],
            template="""
            You are an AI assistant designed to help users understand insurance options based on their needs.
            
            Below is the conversation history:
            {chat_history}
            
            Below is the retrieved data:
            {details}
            
            User Query: {user_input}
            
            Based on the retrieved information, provide a helpful response addressing the user's query.
            If no relevant information is found, provide general guidance and ask clarifying questions.
            Make your response conversational and easy to understand.
            
            Assistant:
            """
        )

        refined_prompt = prompt_template.format(
            chat_history=chat_history_str,
            details=details_str,
            user_input=user_input
        )

        response = self.azure_client.chat.completions.create(
            model=self.gpt_model,
            messages=[
                {"role": "system", "content": "You are an AI assistant."},
                {"role": "user", "content": refined_prompt}
            ],
            temperature=0.7
        )

        assistant_response = response.choices[0].message.content
        self.memory.save_context({"input": user_input}, {"output": assistant_response})

        return assistant_response

    def get_text_embedding(self, text):
        """Get text embedding from Azure OpenAI"""
        if not self.azure_client:
            raise ValueError("Azure OpenAI client not configured.")
        response = self.azure_client.embeddings.create(input=text, model=self.embedding_model)
        return response.data[0].embedding
    
    def list_all_documents(self, collection_name):
        if not self.client.collections.exists(collection_name):
            return []
        
        collection = self.client.collections.get(collection_name)
        results = collection.query.fetch_objects(
            return_properties=["title"]
        )
        
        documents = []
        for result in results.objects:
            documents.append({
                "title": result.properties['title'],
                "id": str(result.uuid)
            })
        
        return documents

In [10]:
class MultilingualVoiceAssistant:
    def __init__(self):
        # Initialize components
        self.recognizer = sr.Recognizer()
        self.translator = googletrans.Translator()
        
        # Initialize Weaviate Manager
        self.weaviate_manager = WeaviateManager(
            WEAVIATE_CONFIG["http_host"],
            WEAVIATE_CONFIG["http_port"],
            WEAVIATE_CONFIG["grpc_host"],
            WEAVIATE_CONFIG["grpc_port"],
            WEAVIATE_CONFIG["secure"],
            AZURE_CONFIG
        )
        
        # User preferences
        self.user_language = "en"  # Default language
        self.input_mode = "text"   # Default input mode
        self.output_mode = "text"  # Default output mode
        
        # Initialize temp directory for audio files
        self.temp_dir = tempfile.mkdtemp()
        
    def setup_preferences(self):
        """Set up user preferences for language and interaction modes"""
        print("\n===== Multilingual Voice Assistant Setup =====")
        
        # Language selection
        print("\nAvailable languages:")
        for i, lang in enumerate(LANGUAGES.keys(), 1):
            print(f"{i}. {lang.capitalize()}")
        
        while True:
            try:
                choice = int(input("\nSelect your preferred language (enter number): "))
                if 1 <= choice <= len(LANGUAGES):
                    self.user_language = list(LANGUAGES.values())[choice-1]
                    break
                else:
                    print("Invalid choice. Please try again.")
            except ValueError:
                print("Please enter a number.")
        
        # Input mode selection
        while True:
            choice = input("\nSelect input mode (voice/text): ").lower()
            if choice in ["voice", "text"]:
                self.input_mode = choice
                break
            else:
                print("Invalid choice. Please enter 'voice' or 'text'.")
        
        # Output mode selection
        while True:
            choice = input("\nSelect output mode (voice/text/both): ").lower()
            if choice in ["voice", "text", "both"]:
                self.output_mode = choice
                break
            else:
                print("Invalid choice. Please enter 'voice', 'text', or 'both'.")
        
        print(f"\nSetup complete! You've selected:")
        print(f"Language: {list(LANGUAGES.keys())[list(LANGUAGES.values()).index(self.user_language)].capitalize()}")
        print(f"Input mode: {self.input_mode.capitalize()}")
        print(f"Output mode: {self.output_mode.capitalize()}")
        print("\n===========================================")
        
        # Initial greeting
        greeting = "Welcome to the Multilingual Insurance Assistant! How can I help you today?"
        self.process_output(greeting)
    
    def voice_to_text(self):
        """Convert voice input to text"""
        with sr.Microphone() as source:
            print("\nListening...")
            self.recognizer.adjust_for_ambient_noise(source, duration=1)
            try:
                audio = self.recognizer.listen(source, timeout=5)
                print("Processing speech...")
                
                # Try to recognize using Google Speech Recognition
                try:
                    # Use the user's selected language for speech recognition
                    text = self.recognizer.recognize_google(audio, language=self.user_language)
                    print(f"You said: {text}")
                    return text
                except sr.UnknownValueError:
                    print("Could not understand audio")
                    return None
                except sr.RequestError as e:
                    print(f"Could not request results; {e}")
                    return None
            except sr.WaitTimeoutError:
                print("No speech detected. Please try again.")
                return None
    
    def detect_language(self, text):
        """Detect language of input text"""
        try:
            return detect(text)
        except:
            # Default to English if detection fails
            return "en"
    
    def translate_text(self, text, src_lang, dest_lang):
        """Translate text from source language to destination language"""
        if src_lang == dest_lang:
            return text
        
        try:
            # Handle source language 'auto' detection
            if src_lang not in googletrans.LANGUAGES and src_lang != 'auto':
                src_lang = 'auto'
                
            # Make sure destination language is supported
            if dest_lang not in googletrans.LANGUAGES:
                print(f"Warning: Unsupported destination language '{dest_lang}'. Defaulting to English.")
                dest_lang = 'en'
                
            translation = self.translator.translate(text, src=src_lang, dest=dest_lang)
            return translation.text
        except Exception as e:
            print(f"Translation error: {e}")
            # Return original text if translation fails
            return text
    
    def text_to_speech(self, text, lang_code):
        """Convert text to speech in specified language"""
        try:
            temp_file = os.path.join(self.temp_dir, f"tts_{uuid.uuid4()}.mp3")
            tts = gTTS(text=text, lang=lang_code, slow=False)
            tts.save(temp_file)
            
            # Play audio
            audio = AudioSegment.from_mp3(temp_file)
            play(audio)
            
            # Clean up file after playing
            os.remove(temp_file)
        except Exception as e:
            print(f"TTS error: {e}")
    
    def process_input(self):
        """Process user input based on selected input mode"""
        if self.input_mode == "voice":
            user_input = self.voice_to_text()
            if not user_input:
                return None
        else:  # text input
            user_input = input("\nYou: ")
        
        if user_input.lower() in ["exit", "quit", "bye"]:
            return "exit"
        
        # Detect language and translate to English if needed
        detected_lang = self.detect_language(user_input)
        
        # Store original text for debugging
        original_text = user_input
        
        # Translate to English for processing if not already in English
        if detected_lang != "en":
            user_input = self.translate_text(user_input, detected_lang, "en")
            print(f"Translated input: {user_input}")
        
        return user_input
    
    def process_output(self, response_text):
        """Process and deliver output based on selected output mode"""
        # Translate response back to user's preferred language if not already
        if self.user_language != "en":
            translated_response = self.translate_text(response_text, "en", self.user_language)
        else:
            translated_response = response_text
        
        # Output as text if selected
        if self.output_mode in ["text", "both"]:
            print(f"\nAssistant: {translated_response}")
        
        # Output as voice if selected
        if self.output_mode in ["voice", "both"]:
            self.text_to_speech(translated_response, self.user_language)
    
    def run(self):
        """Main loop for the assistant"""
        self.setup_preferences()
        
        while True:
            # Get input from user
            user_input = self.process_input()
            
            if user_input == "exit":
                farewell = "Thank you for using the Multilingual Voice Assistant. Goodbye!"
                self.process_output(farewell)
                break
            
            if user_input:
                # Process with RAG and get response
                response = self.weaviate_manager.search_query_in_collections(user_input)
                
                # Deliver response
                self.process_output(response)



In [11]:
assistant = MultilingualVoiceAssistant()

In [12]:
assistant.run()


===== Multilingual Voice Assistant Setup =====

Available languages:
1. English
2. Malay
3. Thai
4. Chinese
5. Tamil
6. Telugu
7. Singlish

Setup complete! You've selected:
Language: English
Input mode: Text
Output mode: Text


Assistant: Welcome to the Multilingual Insurance Assistant! How can I help you today?


RateLimitError: Error code: 429 - {'error': {'code': '429', 'message': 'Requests to the ChatCompletions_Create Operation under Azure OpenAI API version 2023-05-15 have exceeded token rate limit of your current OpenAI S0 pricing tier. Please retry after 60 seconds. Please go here: https://aka.ms/oai/quotaincrease if you would like to further increase the default rate limit. For Free Account customers, upgrade to Pay as you Go here: https://aka.ms/429TrialUpgrade.'}}